# Module 13 Lab - Building ML Pipelines

**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.

**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?

**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.

**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.

**A `scikit-learn` Pipeline solves these problems by:**
1.  **Encapsulating** all workflow steps into a single object.
2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.
3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)

Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features]) # Note: using .transform() here!

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)

print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

/tmp/ipykernel_2423/1388338196.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_2423/1388338196.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" Way

Now, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.

**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
numeric_transformer = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# 2. Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

# 3. Use ColumnTransformer to apply different transformers to different columns
preprocessor = make_column_transformer(
    (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
    (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
)

# 4. Create the final, full pipeline
#    This chains the preprocessor and the final model together.
final_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=42)
)

# 5. Fit and evaluate the entire pipeline in one step!
final_pipeline.fit(X_train, y_train)
y_pred_pipeline = final_pipeline.predict(X_test)

print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

Accuracy (Pipeline Method): 82.68%


## Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

---

**1. Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?

The three biggest advantages of using the Pipeline approach are:

First, **simplicity and cleaner code**. The manual approach requires creating and managing many separate objects — a scaler, an encoder, a model — and manually combining their outputs using `np.hstack`. The Pipeline approach encapsulates all of these steps into a single object called `final_pipeline`, making the code much shorter, easier to read, and easier to maintain.

Second, **automatic prevention of data leakage**. In the Pipeline, all preprocessing steps (imputing, scaling, encoding) are fitted exclusively on the training data when `.fit()` is called, and only applied (transformed) to the test data. This happens automatically without the programmer needing to remember to use `.transform()` instead of `.fit_transform()` on the test set — a common mistake in the manual approach.

Third, **easy deployment and reproducibility**. The entire workflow — preprocessing and model — is saved as a single object. This means you can save the pipeline with `joblib`, load it anywhere, and call `.predict()` on raw new data without needing to manually apply each preprocessing step. This makes deployment to production systems much safer and more reliable.

---

**2. Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?

The distinction is crucial because `fit_transform()` does two things: it learns the scaling parameters (the mean and standard deviation of each feature) from the data AND applies the transformation. If we called `fit_transform()` on the test data, the scaler would learn new parameters from the test set, which means the test data would be scaled differently than the training data. Worse, if we called `fit_transform()` on the entire dataset before splitting, information from the test set (like its mean and standard deviation) would contaminate the training process, making our accuracy estimates overly optimistic.

The Pipeline handles this automatically by design. When you call `final_pipeline.fit(X_train, y_train)`, every transformer in the pipeline calls `fit_transform()` only on the training data. When you later call `final_pipeline.predict(X_test)`, every transformer automatically calls only `transform()` on the test data, using the parameters learned from training. You never have to think about this distinction — the Pipeline enforces correct behavior every time.

---

**3. Extending the Pipeline:** How would you modify your `final_pipeline` object to include a PCA step after scaling and encoding but before the RandomForestClassifier?

To add a PCA step, I would modify the `final_pipeline` by inserting `PCA()` as a step between the `preprocessor` and the `RandomForestClassifier`. The updated pipeline would look like this:

```python
from sklearn.decomposition import PCA

final_pipeline = make_pipeline(
    preprocessor,
    PCA(n_components=10),
    RandomForestClassifier(random_state=42)
)
```

The Pipeline executes steps in order, so PCA would receive the already-scaled and encoded features from the preprocessor, reduce their dimensionality, and then pass the reduced features to the RandomForestClassifier. This is one of the key strengths of Pipelines — adding or removing steps is as simple as inserting or deleting a line.

---

**4. Real-World Value:** Why is giving another team the single `final_pipeline` object much safer than giving them the 5 separate objects from the manual approach?

Giving another team the single `final_pipeline` object is much safer for several important reasons. First, it eliminates the risk of the other team applying the preprocessing steps in the wrong order. If they receive separate objects, they might accidentally scale data before imputing missing values, or forget to encode categorical features entirely — both of which would cause the model to produce incorrect predictions.

Second, with a Pipeline, the other team can call `final_pipeline.predict(raw_data)` directly on new, unprocessed data. The pipeline will automatically apply imputation, scaling, encoding, and prediction in the correct sequence. With separate objects, they would need to carefully apply each step manually in the right order, which is error-prone.

Third, the Pipeline guarantees that the same preprocessing parameters learned during training (the median values for imputation, the mean and standard deviation for scaling, the category encodings) are applied consistently to all new data. If separate objects are passed around, there is a risk that someone might accidentally refit one of them on new data, breaking the consistency between training and deployment. A single Pipeline object is a single source of truth for the entire ML workflow.